# U2 AquaVigía: evidencias de ingesta y semántica de entrega (S6, S7 y S8)

Dimensión U2 de **Celia Patricia Apaza Hilasaca**: temperatura y oxígeno disuelto. Tópico: `agua-usgs-eventos-v2` (3 particiones), alimentado por **MQTT y el puente** (`bridge_mqtt_kafka_usgs.py`).

| Sección | Tema de la evaluación |
|---|---|
| 1 | S6: conceptos de Kafka y contrato de evento versionable |
| 2 | S7: frecuencia y volumen frente al evento de negocio, partición por key, canal MQTT, validación de esquema y rango físico |
| 3 | S8: semántica al menos una vez y deduplicación |
| 4 | S8: ventana fija frente a ventana deslizante, y su efecto en latencia y throughput |

In [1]:
import os, json, time, math, shutil, uuid, urllib.request, urllib.error
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType, MapType

INTEGRANTE = "Patricia Apaza"
ID = "patricia"
VARIABLES = ["temperatura_c", "oxigeno_mgl"]
TOPIC_ENTRADA = "agua-usgs-eventos-v2"
TOPIC_NEGOCIO = "orden-eventos"
BROKER = "kafka:9092"
WATERMARK = "85 minutes"

spark = (SparkSession.builder.appName(f"u2-usgs-temario-{ID}").master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "1").config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4043").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

VARS_CONOCIDAS = ["ph", "oxigeno_mgl", "turbidez_fnu", "conductividad_uscm", "temperatura_c", "nitrato_mgl"]
RANGO_FISICO = {"temperatura_c": (-5, 45), "oxigeno_mgl": (0, 25), "ph": (0, 14),
                "turbidez_fnu": (0, 4000), "conductividad_uscm": (0, 100000), "nitrato_mgl": (0, 100)}

esquema_v2 = (StructType()
    .add("version_esquema", IntegerType())
    .add("evento_id", StringType()).add("estacion_id", StringType()).add("variable", StringType())
    .add("codigo_parametro", StringType()).add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType()).add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType()).add("origen", StringType()).add("fuente", StringType())
    .add("contexto", StructType()
         .add("rezagos", MapType(StringType(), DoubleType()))
         .add("simultaneas", MapType(StringType(), DoubleType()))))

def fuente_batch(topic=None):
    return (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER)
            .option("subscribe", topic or TOPIC_ENTRADA).option("startingOffsets", "earliest")
            .option("includeHeaders", "true").load())

def fuente_stream(inicio="latest", max_por_lote=None):
    r = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BROKER)
         .option("subscribe", TOPIC_ENTRADA).option("startingOffsets", inicio)
         .option("includeHeaders", "true").option("failOnDataLoss", "false"))
    if max_por_lote:
        r = r.option("maxOffsetsPerTrigger", max_por_lote)
    return r.load()

def publicar_kafka(registros, topic=None):
    """registros: lista de (key, value_json)."""
    (spark.createDataFrame(registros, ["key", "value"]).write.format("kafka")
     .option("kafka.bootstrap.servers", BROKER).option("topic", topic or TOPIC_ENTRADA).save())

def parsear(df):
    """Kafka -> columnas tipadas + estado de validacion. No descarta nada: marca."""
    rango = F.lit(False)
    for v, (lo, hi) in RANGO_FISICO.items():
        rango = F.when(F.col("variable") == v, (F.col("valor") >= lo) & (F.col("valor") <= hi)).otherwise(rango)
    d = (df.select(F.col("key").cast("string").alias("key"), "partition", "offset",
                   F.expr("cast(get(filter(headers, h -> h.key = 'canal'), 0).value as string)").alias("canal"),
                   F.expr("cast(get(filter(headers, h -> h.key = 'mqtt_topic'), 0).value as string)").alias("mqtt_topic"),
                   F.col("value").cast("string").alias("json"))
           .withColumn("e", F.from_json("json", esquema_v2)).select("key", "partition", "offset", "canal", "mqtt_topic", "json", "e.*")
           .withColumn("ts", F.to_timestamp("fecha_medicion_utc")))
    esquema_ok = (F.col("estacion_id").isNotNull() & F.col("variable").isNotNull()
                  & F.col("valor").isNotNull() & F.col("ts").isNotNull())
    return d.withColumn("validacion",
        F.when(~esquema_ok, "esquema_invalido")
         .when(~F.col("variable").isin(VARS_CONOCIDAS), "variable_desconocida")
         .when(~rango, "fuera_de_rango_fisico").otherwise("valido"))

todo = parsear(fuente_batch()).cache()
print("mensajes en el topico:", todo.count())

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-83c2b3dd-d1ef-4528-955a-2a07f4f0817b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

mensajes en el topico: 37064


## 1. Kafka y contrato versionable (S6)

- **Tópico** `agua-usgs-eventos-v2`, en un **broker** (`lambda26-kafka`), dividido en **particiones**.
- Cada mensaje tiene un **offset** (posición) dentro de su partición y una **key** (`estacion_id`).
- Spark lee como un **consumer group** propio y guarda los offsets en su checkpoint.
- El contrato es **versionable**: `version_esquema` permite agregar campos sin romper a quien lee la versión anterior.

In [2]:
print("Offsets por particion (posicion de cada mensaje dentro de su particion):")
todo.groupBy("partition").agg(F.count("*").alias("mensajes"), F.min("offset").alias("offset_min"),
                              F.max("offset").alias("offset_max")).orderBy("partition").show()
print("Versiones del contrato en el topico:")
todo.groupBy("version_esquema").count().show()

v2 = todo.where(F.col("version_esquema") == 2).first()
print("Ejemplo de evento version 2:")
print(json.dumps(json.loads(v2["json"]), indent=1, ensure_ascii=False))

esquema_v1 = StructType([f for f in esquema_v2.fields if f.name not in ("version_esquema", "contexto")])
lector_v1 = todo.select(F.from_json("json", esquema_v1).alias("e")).select("e.*")
n_v1_ok = lector_v1.where(F.col("estacion_id").isNotNull() & F.col("valor").isNotNull()).count()
print(f"Un lector de la version 1 entiende {n_v1_ok} de {todo.count()} mensajes: el campo nuevo se ignora.")

Offsets por particion (posicion de cada mensaje dentro de su particion):
+---------+--------+----------+----------+
|partition|mensajes|offset_min|offset_max|
+---------+--------+----------+----------+
|        0|   14692|         0|     14691|
|        1|   14443|         0|     14442|
|        2|    7929|         0|      7928|
+---------+--------+----------+----------+

Versiones del contrato en el topico:
+---------------+-----+
|version_esquema|count|
+---------------+-----+
|              2|37062|
|           NULL|    2|
+---------------+-----+

Ejemplo de evento version 2:
{
 "evento_id": "bffc1a18-943e-4fc3-855f-a81473a7d55b",
 "estacion_id": "USGS-01491000",
 "variable": "ph",
 "codigo_parametro": "00400",
 "valor": 7.3,
 "fecha_medicion_utc": "2026-10-16T22:00:00+00:00",
 "fecha_ingesta_utc": "2026-11-06T06:28:51.502147+00:00",
 "retraso_min": 29308.9,
 "origen": "replay",
 "fuente": "USGS",
 "version_esquema": 2,
 "contexto": {
  "rezagos": {
   "l1": 7.3,
   "l2": 7.3,
   "l

## 2. Ingesta IoT (S7)

### 2.1 Frecuencia y volumen: sensor frente a evento de negocio

In [3]:
mias = todo.where(F.col("variable").isin(VARIABLES) & (F.col("validacion") == "valido"))
w = mias.select("estacion_id", "variable", F.unix_timestamp("ts").alias("t")).distinct()
from pyspark.sql.window import Window
pasos = (w.withColumn("dt_min", (F.col("t") - F.lag("t").over(Window.partitionBy("estacion_id", "variable").orderBy("t"))) / 60)
          .where(F.col("dt_min").isNotNull()))
cad = pasos.approxQuantile("dt_min", [0.5], 0.01)[0]
series = w.select("estacion_id", "variable").distinct().count()
print(f"IoT: {series} series (estacion x variable), una lectura cada {cad:.0f} min -> {series * 1440 / cad:,.0f} eventos/dia")
neg = fuente_batch(TOPIC_NEGOCIO)
n_neg = neg.count()
if n_neg:
    r = neg.agg(F.min("timestamp").alias("a"), F.max("timestamp").alias("b")).first()
    dias = max((r["b"] - r["a"]).total_seconds() / 86400, 1 / 24)
    print(f"Negocio ({TOPIC_NEGOCIO}): {n_neg} eventos en {dias:.2f} dias -> unos {n_neg / dias:,.0f} por dia")
else:
    print(f"Negocio ({TOPIC_NEGOCIO}): aun sin eventos (se generan en el 12c v3)")
print("El sensor es periodico y de alto volumen; el evento de negocio es raro y valioso.")
print("Por eso el IoT va con key por estacion, watermark y deduplicacion, y el negocio con id idempotente.")

IoT: 25 series (estacion x variable), una lectura cada 5 min -> 7,200 eventos/dia
Negocio (orden-eventos): 26 eventos en 7.01 dias -> unos 4 por dia
El sensor es periodico y de alto volumen; el evento de negocio es raro y valioso.
Por eso el IoT va con key por estacion, watermark y deduplicacion, y el negocio con id idempotente.


### 2.2 Partición por dispositivo (key)

Kafka elige la partición con un hash de la key. Con `estacion_id` como key, **cada estación cae siempre en la misma partición** y sus lecturas conservan el orden. Verificación aquí y en Kafka UI (`localhost:48085` → Topics → `agua-usgs-eventos-v2` → Messages: filtrar por key y mirar la columna Partition).

In [4]:
por_key = (todo.where(F.col("key").startswith("USGS-"))
           .groupBy("key").agg(F.countDistinct("partition").alias("particiones_distintas"),
                               F.first("partition").alias("particion"), F.count("*").alias("mensajes")))
por_key.orderBy("particion", "key").show(40, truncate=False)
malas = por_key.where(F.col("particiones_distintas") > 1).count()
print("Estaciones repartidas en mas de una particion:", malas, "(esperado: 0)")
por_key.groupBy("particion").agg(F.count("*").alias("estaciones"), F.sum("mensajes").alias("mensajes")).orderBy("particion").show()

+---------------+---------------------+---------+--------+
|key            |particiones_distintas|particion|mensajes|
+---------------+---------------------+---------+--------+
|USGS-01491000  |1                    |0        |2917    |
|USGS-0164578734|1                    |0        |3119    |
|USGS-0164579522|1                    |0        |2987    |
|USGS-01649500  |1                    |0        |2773    |
|USGS-02074556  |1                    |0        |2896    |
|USGS-01594441  |1                    |1        |2947    |
|USGS-01640152  |1                    |1        |2947    |
|USGS-01646500  |1                    |1        |2917    |
|USGS-01649190  |1                    |1        |2819    |
|USGS-01650800  |1                    |1        |2813    |
|USGS-01579550  |1                    |2        |2939    |
|USGS-01589290  |1                    |2        |2132    |
|USGS-02054550  |1                    |2        |2852    |
+---------------+---------------------+---------+-------

### 2.3 Canal MQTT → puente → Kafka

Un dispositivo real no habla Kafka: el protocolo de Kafka es pesado (conexión TCP persistente, metadatos del clúster, lotes y reintentos) y exigiría exponer el broker a la red de campo. El dispositivo publica por **MQTT**, liviano, y un **puente** junto al clúster lo pasa a Kafka sin tocar el contenido. El puente marca cada mensaje con la cabecera `canal = mqtt`.

In [5]:
todo.groupBy(F.coalesce("canal", F.lit("(sin cabecera: publicado directo)")).alias("canal")).count().show(truncate=False)
todo.where(F.col("canal") == "mqtt").select("mqtt_topic", "key", "partition", "offset").limit(5).show(truncate=False)
print("Topico MQTT por estacion (ultimo segmento = key de Kafka): ver logs con  docker logs lambda26-puente-usgs")

+---------------------------------+-----+
|canal                            |count|
+---------------------------------+-----+
|mqtt                             |36858|
|(sin cabecera: publicado directo)|206  |
+---------------------------------+-----+

+-------------------------------------+-------------+---------+------+
|mqtt_topic                           |key          |partition|offset|
+-------------------------------------+-------------+---------+------+
|aquavigia/equipo02/usgs/USGS-01491000|USGS-01491000|0        |0     |
|aquavigia/equipo02/usgs/USGS-01491000|USGS-01491000|0        |1     |
|aquavigia/equipo02/usgs/USGS-01491000|USGS-01491000|0        |2     |
|aquavigia/equipo02/usgs/USGS-01491000|USGS-01491000|0        |3     |
|aquavigia/equipo02/usgs/USGS-01649500|USGS-01649500|0        |4     |
+-------------------------------------+-------------+---------+------+

Topico MQTT por estacion (ultimo segmento = key de Kafka): ver logs con  docker logs lambda26-puente-usgs


### 2.4 Validación de esquema y de rango físico

Se publican **tres mensajes de prueba** (key `PRUEBA-VALIDACION`, se declaran como prueba): uno con JSON roto, uno sin `valor` y uno con pH 19,5, imposible para el sensor. Spark los clasifica y solo los válidos siguen al modelo.

In [6]:
prueba = [("PRUEBA-VALIDACION", '{"estacion_id": "USGS-PRUEBA", "variable": "ph", "valor": 7.1, "fecha_medicion_utc": '),
          ("PRUEBA-VALIDACION", json.dumps({"version_esquema": 2, "estacion_id": "USGS-PRUEBA", "variable": "ph",
                                            "fecha_medicion_utc": datetime.now(timezone.utc).isoformat()})),
          ("PRUEBA-VALIDACION", json.dumps({"version_esquema": 2, "estacion_id": "USGS-PRUEBA", "variable": "ph", "valor": 19.5,
                                            "fecha_medicion_utc": datetime.now(timezone.utc).isoformat()}))]
publicar_kafka(prueba)
todo.unpersist()
todo = parsear(fuente_batch()).cache()
print("Clasificacion de todo el topico:")
todo.groupBy("validacion").count().orderBy("validacion").show()
print("Los mensajes de prueba:")
todo.where(F.col("key") == "PRUEBA-VALIDACION").select("offset", "variable", "valor", "validacion",
                                                       F.substring("json", 1, 70).alias("json")).show(truncate=False)
print("Rangos fisicos usados:", RANGO_FISICO)

Clasificacion de todo el topico:


[Stage 59:===================>                                      (1 + 2) / 3]

+--------------------+-----+
|          validacion|count|
+--------------------+-----+
|    esquema_invalido|    6|
|fuera_de_rango_fi...|    3|
|              valido|37058|
+--------------------+-----+

Los mensajes de prueba:
+------+--------+-----+---------------------+----------------------------------------------------------------------+
|offset|variable|valor|validacion           |json                                                                  |
+------+--------+-----+---------------------+----------------------------------------------------------------------+
|792   |ph      |19.5 |fuera_de_rango_fisico|{"version_esquema": 2, "estacion_id": "USGS-PRUEBA", "variable": "ph",|
|793   |ph      |NULL |esquema_invalido     |{"version_esquema": 2, "estacion_id": "USGS-PRUEBA", "variable": "ph",|
|794   |NULL    |NULL |esquema_invalido     |{"estacion_id": "USGS-PRUEBA", "variable": "ph", "valor": 7.1, "fecha_|
|5996  |NULL    |NULL |esquema_invalido     |{"estacion_id": "USGS-PRU

## 3. Semántica al menos una vez y deduplicación (S8)

Kafka con Spark garantiza **al menos una vez**: si el productor reintenta o un lote se reprocesa tras un fallo, la misma lectura puede llegar dos veces. Se corren dos consultas sobre lo mismo: una sin deduplicar y otra con `dropDuplicatesWithinWatermark` por (estación, variable, tiempo del evento). Se publican 50 lecturas reales recientes y, después, **las mismas 50 otra vez**.

In [7]:
def contador():
    c = {"filas": 0}
    def f(d, bid): c["filas"] += d.count()
    return c, f

base = parsear(fuente_stream("latest")).where(F.col("validacion") == "valido")
c_sin, f_sin = contador(); c_con, f_con = contador()
q_sin = base.writeStream.foreachBatch(f_sin).option("checkpointLocation", f"/tmp/u2d_sin_{int(time.time())}").trigger(processingTime="3 seconds").start()
q_con = (base.withWatermark("ts", WATERMARK).dropDuplicatesWithinWatermark(["estacion_id", "variable", "ts"])
             .writeStream.foreachBatch(f_con).option("checkpointLocation", f"/tmp/u2d_con_{int(time.time())}")
             .trigger(processingTime="3 seconds").start())
time.sleep(10)
ult = (todo.where(F.col("key").startswith("USGS-") & (F.col("validacion") == "valido"))
           .orderBy(F.desc("offset")).limit(50).select("key", "json").collect())
regs = [(r["key"], r["json"]) for r in ult]
publicar_kafka(regs); time.sleep(12)
antes = (c_sin["filas"], c_con["filas"])
publicar_kafka(regs); time.sleep(12)
q_sin.stop(); q_con.stop()
print(f"Primera vez  -> sin dedup: {antes[0]}   con dedup: {antes[1]}")
print(f"Reenvio igual -> sin dedup: +{c_sin['filas'] - antes[0]}   con dedup: +{c_con['filas'] - antes[1]}  (esperado: +50 y +0)")
for op in (q_con.lastProgress or {}).get("stateOperators", []):
    print("estado de la deduplicacion:", op["operatorName"], "filas:", op["numRowsTotal"],
          "descartados en el ultimo lote:", op.get("customMetrics", {}).get("numDroppedDuplicateRows"))

Primera vez  -> sin dedup: 50   con dedup: 50
Reenvio igual -> sin dedup: +50   con dedup: +0  (esperado: +50 y +0)
estado de la deduplicacion: dedupeWithinWatermark filas: 50 descartados en el ultimo lote: 50


## 4. Ventana fija frente a ventana deslizante (S8)

- **Fija** de 15 min: cada lectura cae en una sola ventana.
- **Deslizante** de 15 min cada 5 min: cada lectura cae en tres ventanas. Da una vista más suave y más actualizada (un resultado nuevo cada 5 min), pero **triplica las filas de salida y el estado** que Spark debe guardar.

Más estado y más filas significan más trabajo por lote: con volumen alto eso sube la latencia y baja el throughput. Abajo se mide con los datos reales de este tópico.

In [8]:
def medir(ancho, paso=None, nombre="x"):
    filas = {"n": 0}
    def f(d, bid): filas["n"] += d.count()
    w = F.window("ts", ancho, paso) if paso else F.window("ts", ancho)
    agg = (parsear(fuente_stream("earliest")).where((F.col("validacion") == "valido") & F.col("variable").isin(VARIABLES))
           .withWatermark("ts", WATERMARK).groupBy(w, "estacion_id", "variable")
           .agg(F.count("*").alias("lecturas"), F.avg("valor").alias("promedio")))
    t0 = time.monotonic()
    q = (agg.writeStream.outputMode("update").foreachBatch(f)
         .option("checkpointLocation", f"/tmp/u2d_ventana_{nombre}_{int(time.time())}")
         .trigger(availableNow=True).start())
    q.awaitTermination()
    seg = time.monotonic() - t0
    prog = [p for p in q.recentProgress if p["numInputRows"] > 0]
    entradas = sum(p["numInputRows"] for p in prog)
    estado = max((op["numRowsTotal"] for p in q.recentProgress for op in p.get("stateOperators", [])), default=0)
    return {"ventana": nombre, "filas_salida": filas["n"], "filas_en_estado": estado,
            "segundos": round(seg, 1), "eventos_por_s": round(entradas / seg, 1)}

res = pd.DataFrame([medir("15 minutes", nombre="fija_15m"), medir("15 minutes", "5 minutes", nombre="deslizante_15m_5m")])
print(res.to_string(index=False))
f, d = res.iloc[0], res.iloc[1]
print(f"\nLa deslizante produce {d.filas_salida / max(f.filas_salida, 1):.1f} veces las filas y guarda "
      f"{d.filas_en_estado / max(f.filas_en_estado, 1):.1f} veces el estado.")
print(f"Tiempo: {f.segundos} s frente a {d.segundos} s ({d.eventos_por_s} frente a {f.eventos_por_s} eventos/s). "
      "Con pocos datos la diferencia de tiempo es chica; lo que escala es el estado.")

          ventana  filas_salida  filas_en_estado  segundos  eventos_por_s
         fija_15m          1645             1645       2.6        14075.4
deslizante_15m_5m          5009             5009       2.4        15516.2

La deslizante produce 3.0 veces las filas y guarda 3.0 veces el estado.
Tiempo: 2.6 s frente a 2.4 s (15516.2 frente a 14075.4 eventos/s). Con pocos datos la diferencia de tiempo es chica; lo que escala es el estado.
